In [0]:
#spark.conf.set("spark.sql.execution.arrow.pyspark.maxRecordsPerBatch", 1)

In [0]:
# Notebook parameters

params = {
    "vol_path": "",
    "proj_dir": "",
    "del_after_extraction": ""
}

# create text widgets
for k in params.keys():
    dbutils.widgets.text(k, "", "")

# fetch values
for k in params.keys():
    params[k] = dbutils.widgets.get(k)
    print(k, ":", params[k])

In [0]:
import os
import time
from datetime import datetime
from tqdm.notebook import tqdm
import pydicom
import asyncio
from functools import lru_cache
from pyspark.sql import types as T
from pyspark.sql import functions as F
from glob import glob

In [0]:
if params["vol_path"] is not None and len(str(params["vol_path"])) > 0:
    ROOT_DIR = params["vol_path"]
else:
    ROOT_DIR = "/Volumes/1_inland/sectra/vone"

full_proj_path = os.path.join(ROOT_DIR, params["proj_dir"])

In [0]:
df = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.dcm")
    .option("recursiveFileLookup", "true")
    .load(full_proj_path)
    .select(["path", "length"])
)


In [0]:
from pyspark.sql.functions import sha2

#df = df.withColumn("sha256", sha2(df["content"], 256))

In [0]:
print(df.count())
display(df.limit(0))

In [0]:
from io import BytesIO
from pydicom.filebase import DicomFileLike

In [0]:
import pandas as pd
from pathlib import Path
import hashlib



def extract_elements(dataset, parent=""):
    elements = []

    for elem in dataset:
        # Build base name (fallback for private/unknown tags)
        name = elem.keyword if elem.keyword else f"Private_{elem.tag.group:04X}_{elem.tag.element:04X}"
        current_path = f"{parent}.{name}" if parent else name

        if elem.VR == "SQ":  # Sequence
            for i, item in enumerate(elem.value):
                item_path = f"{current_path}[{i}]"
                
                # Optional: record the sequence item itself
                elements.append({
                    "path": item_path,
                    "tag": str(elem.tag),
                    "name": name,
                    "vr": "SQ_ITEM",
                    "value": f"Item {i}",
                    "is_private": elem.tag.is_private
                })

                # Recurse into the item
                elements.extend(extract_elements(item, item_path))
        else:
            elements.append({
                "path": current_path,
                "tag": str(elem.tag),
                "name": name,
                "vr": elem.VR,
                "value": str(elem.value),
                "is_private": elem.tag.is_private
            })

    return elements

def process_dicom_batch(pdf_iter):
    DEL_AFTER_EXTRACTION = int(params["del_after_extraction"])
    for pdf in pdf_iter:
        output_rows = []
        
        """
        for path, sha256, content in zip(pdf['path'], pdf['sha256'], pdf['content']):
            try:
                ds = pydicom.dcmread(BytesIO(content), force=True, stop_before_pixels=True)
                #ds.decompress()

            except:
                output_rows.append((path, sha256, None, None, None, None, None, None))   

            
            elements = extract_elements(ds)
            output_rows += [(path, sha256, str(x['path']), str(x['tag']), str(x['name']), str(x['vr']), str(x['value']), str(x['is_private'])) for x in elements]
        """

        for path in pdf['path']:
            elements = None
            try:
                path = str(path).replace("dbfs:", "")

                '''
                sha256 = None
                with open(path, "rb") as f:
                    content = f.read()
                sha256 = hashlib.sha256(content).hexdigest()
                '''
                # Compute SHA256 without loading entire file into memory
                sha256_hash = hashlib.sha256()
                with open(path, "rb") as f:
                    for chunk in iter(lambda: f.read(4096), b""):
                        sha256_hash.update(chunk)

                sha256 = sha256_hash.hexdigest()

                ds = pydicom.dcmread(path,  stop_before_pixels=True)

                elements = extract_elements(ds)

                output_rows += [(path, sha256, str(x['path']), str(x['tag']), str(x['name']), str(x['vr']), str(x['value']), str(x['is_private']), None) for x in elements]


                if DEL_AFTER_EXTRACTION:
                    os.remove(path)

            except Exception as e:
                if elements is None:
                    output_rows.append((path, sha256, None, None, None, None, None, None, str(e)[:1000]))
                else:
                    output_rows += [(path, sha256, str(x['path']), str(x['tag']), str(x['name']), str(x['vr']), str(x['value']), str(x['is_private']), None) for x in elements]


            



                

               


        yield pd.DataFrame(output_rows, columns=["dicom_path", "sha256", "tag_path", "tag", "name", "vr", "value", "is_private", "error_message"])

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, BinaryType

# Define output schema: path + redacted DICOM binary
output_schema = StructType([
    StructField("dicom_path", StringType(), True),
    StructField("sha256", StringType(), True),
    StructField("tag_path", StringType(), True),
    StructField("tag", StringType(), True),
    StructField("name", StringType(), True),
    StructField("vr", StringType(), True),
    StructField("value", StringType(), True),
    StructField("is_private", StringType(), True),
    StructField("error_message", StringType(), True)
])

md_df = df.mapInPandas(process_dicom_batch, schema=output_schema)
md_df = md_df.withColumn("extraction_timestamp", F.current_timestamp())

#display(md_df.limit(10))



In [0]:
#display(md_df.filter("vr = 'SQ_ITEM'").limit(100))




In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS 4_prod.dicom_tags")
spark.sql("""
    CREATE TABLE IF NOT EXISTS 4_prod.dicom_tags.extracted_dicom_tags (
        dicom_path STRING,
        sha256 STRING,
        tag_path STRING,
        tag STRING,
        name STRING,
        vr STRING,
        value STRING,
        is_private STRING,
        error_message STRING,
        extraction_timestamp TIMESTAMP
    )
""")

In [0]:
md_df = md_df.repartition("dicom_path")
md_df.write.mode("append").saveAsTable("4_prod.dicom_tags.extracted_dicom_tags")

In [0]:
%sql

SELECT * FROM 4_prod.dicom_tags.extracted_dicom_tags WHERE extraction_timestamp > DATE(CURRENT_TIMESTAMP()) LIMIT 100